
# Any CSV matches provincial shp by latitude and longitude, and overwrites the original file in place

notebook `ipynb` ,:

1. CSV ;
2. /;
3. shp ;
4. Backfill/overwrite the `ADCODE99` and `NAME_EN_JX` columns into CSV;
5. For **unmatched points**, press **nearest province** to match;
6. ** CSV **( `.bak` ).

## Where do you just need to change?

Just need to change the following parameters in the unit:

```python
input_csv = r"csv"
```

The rest can be run directly.

## Matching logic

- First use `within` to do strict in-plane matching;
- For points that are not matched by `within`, use `nearest` to make the nearest match;
- In this way, boundary points and slight offset points can also be added.

## Description

- If there is `NAME_EN_JX` field in shp, the script will use it directly;
- If your current shp does not have `NAME_EN_JX`, the script will give priority to automatically supplement the standard province name based on `ADCODE99`;
- If it still cannot be filled, the name field in shp will be returned.


In [ ]:

import os
import re
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=UserWarning)

print("Dependency import completed.")
print("pandas version:", pd.__version__)
print("geopandas version:", gpd.__version__)


In [ ]:

# =========================
# Parameter area: You only need to change input_csv
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
# Just change it to your CSV path.
# input_csv = os.path.join(base_path, "data", "output", "Positive_Negative_balanced_25366.csv")
input_csv = os.path.join(base_path, "data", "output", "Points_China_all_10km.csv")
# If the longitude and latitude column names are very special and automatic recognition fails, you can specify them manually; otherwise, keep None
lon_col = None
lat_col = None

# Whether to back up the original file first (a .bak file with the same name will be generated)
backup_original = True

# Whether to write the matching method back to CSV
# False: / ADCODE99 NAME_EN_JX
# True: additionally write the province_match_method column to facilitate verification
save_match_method_column = False

# Output encoding: None means using the original CSV encoding as much as possible
# Excel , "utf-8-sig"
output_encoding = None


# =========================================
# Provincial shp path: Prioritize the path writing of your original notebook
# If it cannot be found, the script will automatically search in the same directory/current directory of the CSV.
# =========================================

province_no_TW_AM_HK_shp = os.path.join(
    base_path, "data", "Administrative_divisions_of_china",
    "no_TW_AM_HK", "china_pro2_no_TW_AM_HK_geographical_division.shp"
)

print("input_csv =", input_csv)
print("province_no_TW_AM_HK_shp =", province_no_TW_AM_HK_shp)


In [ ]:

# Processing step.

PROVINCE_NAME_EN_FALLBACK = {
    110000: "Beijing",
    120000: "Tianjin",
    130000: "Hebei",
    140000: "Shanxi",
    150000: "Inner Mongolia",
    210000: "Liaoning",
    220000: "Jilin",
    230000: "Heilongjiang",
    310000: "Shanghai",
    320000: "Jiangsu",
    330000: "Zhejiang",
    340000: "Anhui",
    350000: "Fujian",
    360000: "Jiangxi",
    370000: "Shandong",
    410000: "Henan",
    420000: "Hubei",
    430000: "Hunan",
    440000: "Guangdong",
    450000: "Guangxi",
    460000: "Hainan",
    500000: "Chongqing",
    510000: "Sichuan",
    520000: "Guizhou",
    530000: "Yunnan",
    540000: "Tibet",
    610000: "Shaanxi",
    620000: "Gansu",
    630000: "Qinghai",
    640000: "Ningxia",
    650000: "Xinjiang",
}


def read_csv_with_fallback(path, **kwargs):
    """CSV."""
    encodings = ["utf-8", "utf-8-sig", "gb18030", "gbk"]
    errors = []

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            return df, enc
        except Exception as e:
            errors.append(f"{enc}: {type(e).__name__}: {e}")

    try:
        df = pd.read_csv(path, encoding="latin1", **kwargs)
        print("⚠️ Using latin1 to read from the bottom, Chinese characters may be garbled, please check the original CSV encoding.")
        return df, "latin1"
    except Exception:
        raise RuntimeError("CSV read failed. Encoding tried: \\n" + "\n".join(errors))


def normalize_col(name):
    return re.sub(r"[^0-9a-zA-Z\u4e00-\u9fff]+", "", str(name)).lower()


def detect_column(df, manual_col, candidates, fuzzy_tokens, label):
    """;."""
    if manual_col is not None:
        if manual_col not in df.columns:
            raise KeyError(f"{label} column does not exist: {manual_col}")
        return manual_col

    norm_map = {normalize_col(col): col for col in df.columns}

    for cand in candidates:
        cand_norm = normalize_col(cand)
        if cand_norm in norm_map:
            return norm_map[cand_norm]

    matched = []
    for col in df.columns:
        norm = normalize_col(col)
        if any(token in norm for token in fuzzy_tokens):
            matched.append(col)

    if len(matched) == 1:
        return matched[0]

    if len(matched) > 1:
        matched = sorted(matched, key=lambda x: (len(normalize_col(x)), str(x)))
        return matched[0]

    raise KeyError(
        f"Not found{label}.\n"
        f"Current CSV column names:{df.columns.tolist()}\n"
        f"Could not find a {label} column name."
    )


def resolve_province_shp(input_csv, preferred_path=None):
    """; shp."""
    candidates = []

    if preferred_path:
        candidates.append(Path(preferred_path).expanduser())

    csv_dir = Path(input_csv).expanduser().resolve().parent
    cwd = Path.cwd()

    candidate_names = [
        "china_pro2_no_TW_AM_HK_geographical_division.shp",
        "china_pro2_no_TW_AM_HK.shp",
        os.path.join(
            "data", "Administrative_divisions_of_china", "no_TW_AM_HK",
            "china_pro2_no_TW_AM_HK_geographical_division.shp"
        ),
        os.path.join(
            "data", "Administrative_divisions_of_china", "no_TW_AM_HK",
            "china_pro2_no_TW_AM_HK.shp"
        ),
    ]

    for base in [csv_dir, cwd]:
        for name in candidate_names:
            candidates.append(base / name)

    deduped = []
    seen = set()
    for p in candidates:
        sp = str(p)
        if sp not in seen:
            deduped.append(p)
            seen.add(sp)

    for p in deduped:
        if p.exists():
            return str(p.resolve())

    raise FileNotFoundError(
        "Provincial shp not found, please check the path. Tried: \\n" +
        "\n".join(str(p) for p in deduped)
    )


def choose_metric_work_crs(gdf):
    """, CRS , EPSG:3857."""
    if gdf.crs is None:
        raise ValueError("shp is missing CRS information and cannot perform spatial matching.")

    try:
        is_geographic = bool(gdf.crs.is_geographic)
    except Exception:
        is_geographic = False

    return "EPSG:3857" if is_geographic else gdf.crs


def detect_shp_fields(province_gdf):
    code_col = detect_column(
        province_gdf,
        manual_col=None,
        candidates=["ADCODE99", "ADCODE", "adcode99", "adcode"],
        fuzzy_tokens=["adcode", "code"],
        label="Province code"
    )

    name_col = detect_column(
        province_gdf,
        manual_col=None,
        candidates=["NAME_EN_JX", "NAME_EN", "NAME", "PROVINCE", "Province"],
        fuzzy_tokens=["nameenjx", "nameen", "province", "name"],
        label="Province name"
    )

    return code_col, name_col


def nearest_join_fallback(points_unmatched, province_work):
    """sjoin_nearest ,."""
    idx_pairs, distances = province_work.sindex.nearest(
        points_unmatched.geometry,
        return_all=False,
        return_distance=True,
    )

    left_rows = points_unmatched.iloc[idx_pairs[0]].reset_index(drop=True)
    right_rows = province_work.iloc[idx_pairs[1]][["ADCODE99", "NAME_EN_JX"]].reset_index(drop=True)

    out = pd.concat([left_rows[["_row_id"]], right_rows], axis=1)
    out["distance_to_province"] = distances
    return out


def standardize_name_en_jx(province_gdf, original_name_field):
    """If shp lacks NAME_EN_JX, priority will be given to ADCODE99 to automatically fill in the standard English province name."""
    province_gdf = province_gdf.copy()
    adc_num = pd.to_numeric(province_gdf["ADCODE99"], errors="coerce").astype("Int64")

    if original_name_field != "NAME_EN_JX":
        mapped = adc_num.map(PROVINCE_NAME_EN_FALLBACK)
        province_gdf["NAME_EN_JX"] = mapped.fillna(province_gdf["NAME_EN_JX"])

    return province_gdf


In [ ]:

# Main function: match and overwrite CSV in place

def add_province_fields_to_csv(
    input_csv,
    province_shp_path=None,
    lon_col=None,
    lat_col=None,
    backup_original=True,
    save_match_method_column=False,
    output_encoding=None,
):
    input_csv = str(Path(input_csv).expanduser().resolve())

    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"CSV:{input_csv}")

    province_shp_path = resolve_province_shp(input_csv, province_shp_path)

    print("=" * 80)
    print("Enter CSV:", input_csv)
    print("Provincial shp:", province_shp_path)
    print("=" * 80)

    # 1) CSV
    df, read_encoding = read_csv_with_fallback(input_csv)
    print(f"CSV encoding:{read_encoding}")
    print(f"CSV / :{df.shape}")

    # 2) Automatically identify latitude and longitude columns
    lon_candidates = [
        "Longitude", "longitude", "LONGITUDE", "lon", "Lon", "LON",
        "lng", "Lng", "LNG", "Longitude", "jingdu"
    ]
    lat_candidates = [
        "Latitude", "latitude", "LATITUDE", "lat", "Lat", "LAT",
        "Latitude", "weidu"
    ]

    lon_col = detect_column(
        df, lon_col,
        candidates=lon_candidates,
        fuzzy_tokens=["longitude", "lon", "lng", "Longitude", "jingdu"],
        label="Longitude"
    )
    lat_col = detect_column(
        df, lat_col,
        candidates=lat_candidates,
        fuzzy_tokens=["latitude", "lat", "Latitude", "weidu"],
        label="Latitude"
    )

    print(f"Longitude column recognized:{lon_col}")
    print(f"Recognized the latitude column:{lat_col}")

    df = df.copy()
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")

    invalid_range_mask = (
        df[lon_col].notna() & ((df[lon_col] < -180) | (df[lon_col] > 180))
    ) | (
        df[lat_col].notna() & ((df[lat_col] < -90) | (df[lat_col] > 90))
    )

    if invalid_range_mask.any():
        print(f"⚠️{int(invalid_range_mask.sum())}The row exceeds the latitude and longitude range and has been processed as missing coordinates.")
        df.loc[invalid_range_mask, [lon_col, lat_col]] = np.nan

    df["_row_id"] = np.arange(len(df))
    valid_coord_mask = df[lon_col].notna() & df[lat_col].notna()

    print(f":{int(valid_coord_mask.sum())} / {len(df)}")
    print(f"Missing latitude and longitude points:{int((~valid_coord_mask).sum())}")

    # 3) Read provincial shp
    province_gdf = gpd.read_file(province_shp_path)
    code_src_col, name_src_col = detect_shp_fields(province_gdf)

    print("Provincial boundary CRS:", province_gdf.crs)
    print(f"Province code field:{code_src_col}")
    print(f":{name_src_col}")

    province = province_gdf[[code_src_col, name_src_col, "geometry"]].copy()
    province = province.rename(columns={code_src_col: "ADCODE99", name_src_col: "NAME_EN_JX"})
    province = province[province.geometry.notna()].copy()
    province = standardize_name_en_jx(province, name_src_col)

    if name_src_col != "NAME_EN_JX":
        print(
            f"⚠️ There is no NAME_EN_JX field in shp,"
            f"has given priority to ADCODE99 to automatically complete the standard English province name; return it if necessary{name_src_col}."
        )

    # 4) Spatial matching: first within, then nearest
    if valid_coord_mask.sum() == 0:
        joined_final = pd.DataFrame(
            columns=["_row_id", "ADCODE99", "NAME_EN_JX", "province_match_method", "distance_to_province"]
        )
        print("No valid latitude and longitude for spatial matching.")
    else:
        points_df = df.loc[valid_coord_mask, ["_row_id", lon_col, lat_col]].copy()
        points_gdf = gpd.GeoDataFrame(
            points_df,
            geometry=gpd.points_from_xy(points_df[lon_col], points_df[lat_col]),
            crs="EPSG:4326",
        )

        work_crs = choose_metric_work_crs(province)
        province_work = province.to_crs(work_crs) if str(province.crs) != str(work_crs) else province.copy()
        points_work = points_gdf.to_crs(work_crs)

        # 4.1 Strict in-plane matching
        joined = gpd.sjoin(
            points_work,
            province_work,
            how="left",
            predicate="within"
        )

        joined = (
            joined.sort_values("_row_id")
                  .drop_duplicates(subset="_row_id", keep="first")
                  .set_index("_row_id")
        )

        joined["province_match_method"] = np.where(joined["ADCODE99"].notna(), "within", "unmatched")
        joined["distance_to_province"] = np.where(joined["ADCODE99"].notna(), 0.0, np.nan)

        unmatched_ids = joined.index[joined["ADCODE99"].isna()]
        print("within Unmatched points:", int(len(unmatched_ids)))

        # 4.2 Make the nearest match for unmatched points
        if len(unmatched_ids) > 0:
            points_unmatched = points_work[points_work["_row_id"].isin(unmatched_ids)].copy()

            try:
                nearest = gpd.sjoin_nearest(
                    points_unmatched,
                    province_work,
                    how="left",
                    distance_col="distance_to_province"
                )
                nearest = (
                    nearest.sort_values(["_row_id", "distance_to_province"])
                           .drop_duplicates(subset="_row_id", keep="first")
                           .set_index("_row_id")
                )
            except Exception as e:
                print("⚠️ The execution of sjoin_nearest fails, use sindex.nearest instead to find out.")
                print("Error message:", e)
                nearest = nearest_join_fallback(points_unmatched, province_work).set_index("_row_id")

            fill_ids = nearest.index[nearest["ADCODE99"].notna()]

            joined.loc[fill_ids, "ADCODE99"] = nearest.loc[fill_ids, "ADCODE99"]
            joined.loc[fill_ids, "NAME_EN_JX"] = nearest.loc[fill_ids, "NAME_EN_JX"]
            joined.loc[fill_ids, "distance_to_province"] = nearest.loc[fill_ids, "distance_to_province"].values
            joined.loc[fill_ids, "province_match_method"] = np.where(
                nearest.loc[fill_ids, "distance_to_province"].fillna(0).eq(0),
                "nearest_0m",
                "nearest"
            )

        joined_final = joined.reset_index()[[
            "_row_id", "ADCODE99", "NAME_EN_JX", "province_match_method", "distance_to_province"
        ]]

        print("Matching method statistics:")
        display(joined_final["province_match_method"].value_counts(dropna=False))
        print("Still has not matched the points of the province:", int(joined_final["ADCODE99"].isna().sum()))

    # 5)
    result_df = df.copy()

    # Force overwrite/create new target column
    result_df["ADCODE99"] = pd.NA
    result_df["NAME_EN_JX"] = pd.NA

    if save_match_method_column:
        result_df["province_match_method"] = "missing_coordinates"

    if len(joined_final) > 0:
        result_df.loc[joined_final["_row_id"], "ADCODE99"] = joined_final["ADCODE99"].values
        result_df.loc[joined_final["_row_id"], "NAME_EN_JX"] = joined_final["NAME_EN_JX"].values

        if save_match_method_column:
            result_df.loc[joined_final["_row_id"], "province_match_method"] = (
                joined_final["province_match_method"].fillna("unmatched").values
            )

    if save_match_method_column:
        result_df.loc[~valid_coord_mask, "province_match_method"] = "missing_coordinates"

    # Processing step.
    try:
        result_df["ADCODE99"] = pd.to_numeric(result_df["ADCODE99"], errors="coerce").astype("Int64")
    except Exception:
        pass

    # Processing step.
    result_df = result_df.drop(columns=["_row_id"])

    # 6) Overwrite CSV in place (backup first, then replace with temporary file)
    if backup_original:
        backup_path = input_csv + ".bak"
        shutil.copy2(input_csv, backup_path)
        print("The original file has been backed up:", backup_path)

    tmp_output = input_csv + ".tmp"
    write_encoding = output_encoding or read_encoding
    if write_encoding == "latin1":
        write_encoding = "utf-8-sig"

    result_df.to_csv(tmp_output, index=False, encoding=write_encoding)
    os.replace(tmp_output, input_csv)

    print("✅ Original CSV has been overwritten:", input_csv)
    print("Total number of lines:", len(result_df))
    print(":", int(result_df["ADCODE99"].notna().sum()))
    print("Number of unmatched lines:", int(result_df["ADCODE99"].isna().sum()))

    return result_df


In [ ]:

# Execute

result_df = add_province_fields_to_csv(
    input_csv=input_csv,
    province_shp_path=province_no_TW_AM_HK_shp,
    lon_col=lon_col,
    lat_col=lat_col,
    backup_original=backup_original,
    save_match_method_column=save_match_method_column,
    output_encoding=output_encoding,
)

print("\\n, 5 :")
display(result_df.head())


In [ ]:

# :
# Only when save_match_method_column = True, this column will display the matching method.

if "province_match_method" not in result_df.columns:
    print("Currently save_match_method_column = False, the province_match_method column is not written out.")
else:
    review_df = result_df[result_df["province_match_method"] != "within"].copy()
    if len(review_df) == 0:
        print("All valid coordinate points are directly matched through within.")
    else:
        print(f"Number of samples to be reviewed:{len(review_df)}")
        show_cols = [c for c in [lon_col, lat_col, "ADCODE99", "NAME_EN_JX", "province_match_method"] if c in review_df.columns]
        display(review_df[show_cols].head(20))
